# 🚀 Whisper & Video Transcriber Studio (Colab Pro Edition)

### Complete All-in-One Modern Studio for Video to Audio & faster-whisper

**Features:**
- 🎬 **FFmpeg Video to Audio Extraction**: Speech-optimized MP3 (32kbps/16kHz mono), WAV, AAC, FLAC, Opus.
- 🎙️ **GPU-Accelerated faster-whisper**: `large-v3`, `distil-large-v3` (6x speed), `large-v2`, `medium`.
- ⚡ **One-Click End-to-End Pipeline**: Feed videos ➤ extract audio ➤ batch transcribe ➤ generate `.txt`, `.srt`, and `.md` Study Notes.
- 📄 **Multi-Format Transcripts**: Timestamped lines (`.txt`), subtitle timings (`.srt`), and readable formatted paragraph study notes (`.md`).
- 🎨 **Modern Glassmorphism UI/UX**: Interactive web UI rendered directly inside your Colab cell or popped out into a dedicated tab.
- 🔄 **100% Resumable**: Automatically skips finished files; reconnecting never re-transcribes completed sessions.
- 👀 **Built-in Transcripts Studio**: Read notes with formatted timestamps, copy to clipboard, or inspect subtitles right in the browser.

---
### ⚡ Quick Start
1. **Runtime → Change runtime type → GPU** (Choose **A100**, **L4**, or **T4**).
2. (Optional) Add your Hugging Face token in **Colab Secrets** (🔑 icon on left) named `HF_TOKEN` with Notebook access ON.
3. Run **Step 1** (Setup & Mount Drive), then run **Step 2** to launch the interactive Studio!

## Step 1: One-Click Environment Setup & Drive Mount
Installs PyAV 18.1.0, faster-whisper, and CUDA 12 libraries, then mounts Google Drive.

In [ ]:
# @title 🔧 Install Dependencies & Mount Drive
!pip install -q -U faster-whisper "av==18.1.0" nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import os, sys, glob, ctypes, logging
from pathlib import Path

# 1. Load CUDA 12 libraries
try:
    import nvidia.cublas.lib, nvidia.cudnn.lib
    libs = []
    for pkg in (nvidia.cublas.lib, nvidia.cudnn.lib):
        d = list(pkg.__path__)[0]
        libs += sorted(glob.glob(os.path.join(d, "*.so*")))
    for _ in range(2):
        for so in libs:
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
    ctypes.CDLL("libcublas.so.12")
    print("✅ CUDA 12 libraries loaded OK")
except Exception as e:
    print(f"⚠️ CUDA 12 load note: {e}")

# 2. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted at /content/drive")
except Exception as e:
    print(f"Note: Drive mount skipped ({e})")

# 3. Optional HF Token
try:
    from google.colab import userdata
    hf = userdata.get("HF_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
        print("✅ HF_TOKEN loaded from Colab Secrets.")
except Exception:
    logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

# 4. Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## Step 2: Launch Interactive Studio Web UI
Launches the modern all-in-one Web UI directly inside this notebook cell, with a button to pop it into a full browser window.

In [ ]:
# @title 🎛️ Launch Whisper & Video Studio
import os, zlib, base64
from pathlib import Path

# Self-extract colab_studio.py if not already present
if not Path("colab_studio.py").exists():
    drive_studio = Path("/content/drive/MyDrive/colab_studio.py")
    if drive_studio.exists():
        import shutil
        shutil.copy(drive_studio, "colab_studio.py")
    else:
        payload = "..."
        Path("colab_studio.py").write_bytes(zlib.decompress(base64.b64decode(payload)))
        print("✅ colab_studio.py unpacked successfully!")

# Launch studio inside notebook cell
from colab_studio import launch_in_colab
launch_in_colab(port=8765, height=950)

## Step 3: (Optional) Headless Batch Transcriber
If you want to run batch transcription in the background without keeping the interactive web UI open, configure the form below and run this cell.

In [ ]:
# @title ⚙️ Headless Batch Configuration
from pathlib import Path
import os, re, time, shutil
from faster_whisper import WhisperModel, BatchedInferencePipeline

# ---------------- SETTINGS ----------------
AUDIO_DIR     = "/content/drive/MyDrive/transcribe/audio" #@param {type:"string"}
OUT_DIR       = "/content/drive/MyDrive/transcribe/transcripts_large_v3" #@param {type:"string"}
MODEL_NAME    = "large-v3" #@param ["large-v3", "distil-large-v3", "large-v2", "medium", "small"]
LANGUAGE      = "en" #@param {type:"string"}
COMPUTE_TYPE  = "float16" #@param ["float16", "int8_float16", "int8", "bfloat16"]
USE_BATCHED   = True #@param {type:"boolean"}
BATCH_SIZE    = 16 #@param {type:"integer"}
PARA_SECONDS  = 45 #@param {type:"integer"}
SESSION_START = 1 #@param {type:"integer"}
SESSION_END   = 999 #@param {type:"integer"}
MAX_HOURS     = 10 #@param {type:"number"}
HOTWORDS      = "Naresh IT, Veera Babu, Kubernetes, kubectl, Terraform, AWS, EC2, S3, IAM, VPC, Docker, Jenkins, Helm, CI/CD, GitHub Actions, Ansible, Prometheus, Grafana, Azure, GCP, YAML, DevOps, SRE" #@param {type:"string"}
# ------------------------------------------

audio_path = Path(AUDIO_DIR)
out_path = Path(OUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

KEYWORD_RE = re.compile(r"\b(?:session|day|class|lecture|lesson|part|episode|ep|video|module|chapter|week)[\s._#-]*(\d+)", re.I)
def file_num(stem):
    m = KEYWORD_RE.search(stem)
    if m: return int(m.group(1))
    m = re.search(r"\d+", stem)
    return int(m.group()) if m else None

def natural(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def ts(sec, srt=False):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    if srt:
        ms = int((sec - int(sec)) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    return f"{h:02d}:{m:02d}:{s:02d}"

video_exts = {".mp4", ".mkv", ".webm", ".avi", ".mov", ".flv", ".wmv", ".m4v", ".ts", ".mts", ".m2ts", ".mpg", ".mpeg", ".3gp", ".3g2", ".ogv", ".vob", ".rm", ".rmvb", ".asf", ".divx", ".f4v", ".m2v", ".mpv", ".qt", ".tod", ".mod"}
audio_exts = {".mp3", ".m4a", ".wav", ".aac", ".flac", ".ogg", ".opus", ".wma", ".aiff", ".aif", ".aifc", ".alac", ".mka", ".ac3", ".dts", ".amr", ".caf", ".mp2", ".m4b", ".m4p", ".weba"}
exts = video_exts | audio_exts
files = [p for p in audio_path.iterdir() if p.suffix.lower() in exts]
files.sort(key=lambda p: (file_num(p.stem) is None, file_num(p.stem) or 0, natural(p.name)))

# Filter by range
files = [p for p in files if (file_num(p.stem) is None) or (SESSION_START <= file_num(p.stem) <= SESSION_END)]
todo = [p for p in files if not (out_path / f"{p.stem}.txt").exists()]

print(f"📊 Total matching files: {len(files)} | Already finished: {len(files)-len(todo)} | Remaining to transcribe: {len(todo)}")

if todo:
    print(f"Loading WhisperModel ({MODEL_NAME} on cuda / {COMPUTE_TYPE})...")
    model = WhisperModel(MODEL_NAME, device="cuda", compute_type=COMPUTE_TYPE)
    batched = BatchedInferencePipeline(model=model) if USE_BATCHED else None
    T0 = time.time()
    ok, failed, total_audio_min = 0, [], 0.0

    for n, f in enumerate(todo, 1):
        if MAX_HOURS and (time.time() - T0) > MAX_HOURS * 3600:
            print(f"\n⏰ Reached MAX_HOURS ({MAX_HOURS} h). Clean stop.")
            break
        
        t0 = time.time()
        is_aud = f.suffix.lower() in audio_exts
        local = Path("/content") / (f.name if is_aud else f"{f.stem}.mp3")
        try:
            if is_aud:
                shutil.copy(f, local)  # Direct audio file, no conversion needed
            else:
                import subprocess
                subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(f), "-vn", "-map", "0:a:0", "-ac", "1", "-ar", "16000", "-b:a", "32k", str(local)], check=True)
            
            if USE_BATCHED:
                segments, info = batched.transcribe(str(local), language=LANGUAGE, batch_size=BATCH_SIZE, beam_size=5, initial_prompt=HOTWORDS)
            else:
                segments, info = model.transcribe(str(local), language=LANGUAGE, vad_filter=True, beam_size=5, condition_on_previous_text=False, initial_prompt=HOTWORDS)
            
            txt_tmp = out_path / f"{f.stem}.txt.part"
            srt_tmp = out_path / f"{f.stem}.srt.part"
            md_tmp  = out_path / f"{f.stem}.md.part"
            paras = []

            with open(txt_tmp, "w", encoding="utf-8") as ft, open(srt_tmp, "w", encoding="utf-8") as fs:
                for i, s in enumerate(segments, 1):
                    text = s.text.strip()
                    ft.write(f"[{ts(s.start)}] {text}\n")
                    fs.write(f"{i}\n{ts(s.start, True)} --> {ts(s.end, True)}\n{text}\n\n")
                    if not paras or s.start - paras[-1][0] >= PARA_SECONDS:
                        paras.append([s.start, []])
                    paras[-1][1].append(text)

            with open(md_tmp, "w", encoding="utf-8") as fm:
                fm.write(f"# {f.stem}\n\n> **Duration:** {ts(info.duration)} | **Model:** {MODEL_NAME}\n\n---\n\n")
                for start_sec, texts in paras:
                    fm.write(f"**[{ts(start_sec)}]** " + " ".join(texts) + "\n\n")

            srt_tmp.rename(out_path / f"{f.stem}.srt")
            md_tmp.rename(out_path / f"{f.stem}.md")
            txt_tmp.rename(out_path / f"{f.stem}.txt")

            took = time.time() - t0
            dur = info.duration
            ok += 1
            total_audio_min += dur / 60
            avg_took = (time.time() - T0) / ok
            left = len(todo) - n
            print(f"[{n}/{len(todo)}] ✅ {f.name} | {dur/60:.1f} min audio | took {took/60:.1f} min ({dur/took:.1f}x) | ETA: {left*avg_took/3600:.1f} h left")
        except Exception as e:
            failed.append(f.name)
            print(f"[{n}/{len(todo)}] ❌ FAILED: {f.name} -> {e}")
        finally:
            local.unlink(missing_ok=True)

    print("\n================ SUMMARY ================")
    print(f"Finished: {ok} file(s) | {total_audio_min/60:.1f} h audio in {(time.time()-T0)/3600:.2f} h total time")
    print(f"Failed: {len(failed)} {failed if failed else ''}")
    remaining = len([p for p in files if not (out_path / f"{p.stem}.txt").exists()])
    print(f"Remaining: {remaining}")
else:
    print("🎉 All files in the selected range are already transcribed!")

## Step 4: Spot-Check Recent Transcripts
Quickly print the top of the most recently generated `.txt` transcript and `.md` study notes.

In [ ]:
# @title 🔍 Preview Latest Transcript
from pathlib import Path
out_dir = Path("/content/drive/MyDrive/transcribe/transcripts_large_v3")
latest = max(out_dir.glob("*.txt"), key=lambda p: p.stat().st_mtime, default=None)
if latest:
    print(f"📄 Latest File: {latest.name}\n")
    print("".join(open(latest, encoding="utf-8").readlines()[:25]))
    md_file = out_dir / f"{latest.stem}.md"
    if md_file.exists():
        print(f"\n📖 Study Notes Preview ({md_file.name}):\n")
        print("".join(open(md_file, encoding="utf-8").readlines()[:20]))
else:
    print("No transcripts found yet in", out_dir)